# 3. MITMA: Silver → Gold

Esquema y descripción de las tablas: [gold.md](../docs/gold.md).

In [38]:
from pathlib import Path
import duckdb
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DB = ROOT / 'data' / 'processed' / 'mitma.duckdb'
con = duckdb.connect(str(DB))
con.execute("SET memory_limit='80%'")
con.execute('CREATE SCHEMA IF NOT EXISTS gold')

In [39]:
con.execute('LOAD spatial')
con.execute('DROP VIEW IF EXISTS gold.mapa_movilidad_diaria')
# Recrear únicamente las tablas generadas, primero las dependientes.
for tabla in ['movilidad_diaria_zona', 'od_horaria', 'zonas']:
    con.execute(f'DROP TABLE IF EXISTS gold.{tabla}')

## Zonas

In [40]:
con.execute("""CREATE TABLE gold.zonas (
    nivel VARCHAR,
    zona VARCHAR,
    nombre VARCHAR,
    poblacion_referencia DOUBLE,
    archivo_nombres VARCHAR,
    archivo_poblacion VARCHAR,
    en_catalogo BOOLEAN,
    geometria GEOMETRY('EPSG:4326'),
    geometria_valida BOOLEAN,
    centroide GEOMETRY('EPSG:4326'),
    PRIMARY KEY (nivel, zona)
)""")

con.execute("""INSERT INTO gold.zonas
SELECT z.*, g.geometria, g.geometria_valida, g.centroide
    FROM silver.zonas z LEFT JOIN silver.geografia_zonas g USING(nivel, zona)""")


## Flujos origen-destino horarios

In [41]:
con.execute("""CREATE TABLE gold.od_horaria (
    nivel VARCHAR,
    fecha DATE,
    hora SMALLINT,
    origen VARCHAR,
    destino VARCHAR,
    viajes_observados DOUBLE,
    viajes_km_observados DOUBLE,
    PRIMARY KEY (nivel, fecha, hora, origen, destino),
    FOREIGN KEY (nivel, origen) REFERENCES gold.zonas (nivel, zona),
    FOREIGN KEY (nivel, destino) REFERENCES gold.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO gold.od_horaria
SELECT nivel, fecha, hora, origen, destino,
           sum(viajes) AS viajes_observados, sum(viajes_km) AS viajes_km_observados,
    FROM silver.viajes GROUP BY nivel, fecha, hora, origen, destino""")
display(con.sql('SELECT * FROM gold.od_horaria ORDER BY viajes_observados DESC LIMIT 10').df())


,nivel,fecha,hora,origen,destino,viajes_observados,viajes_km_observados
0,GAU,2023-01-01,18,GAU Madrid,GAU Madrid,758481.252967,3.849628e+06
1,GAU,2023-01-01,14,GAU Madrid,GAU Madrid,743153.733251,3.481778e+06
2,GAU,2023-01-01,13,GAU Madrid,GAU Madrid,741623.009327,3.662880e+06
3,GAU,2023-01-01,13,GAU Barcelona,GAU Barcelona,704576.608257,3.372462e+06
4,GAU,2023-01-01,19,GAU Madrid,GAU Madrid,702825.683386,3.553042e+06
5,GAU,2023-01-01,20,GAU Madrid,GAU Madrid,687021.663126,3.028820e+06
6,GAU,2023-01-01,18,GAU Barcelona,GAU Barcelona,662006.073297,3.038790e+06
7,GAU,2023-01-01,17,GAU Madrid,GAU Madrid,657134.476619,3.359158e+06
8,GAU,2023-01-01,21,GAU Madrid,GAU Madrid,623152.830521,2.567209e+06
9,GAU,2023-01-01,14,GAU Barcelona,GAU Barcelona,611042.918928,2.525729e+06


## Movilidad diaria por zona

In [42]:
con.execute("""CREATE TABLE gold.movilidad_diaria_zona (
    nivel VARCHAR,
    fecha DATE,
    zona VARCHAR,
    viajes_emitidos DOUBLE,
    viajes_recibidos DOUBLE,
    PRIMARY KEY (nivel, fecha, zona),
    FOREIGN KEY (nivel, zona) REFERENCES gold.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO gold.movilidad_diaria_zona
WITH emitidos AS (
        SELECT nivel, fecha, origen AS zona, sum(viajes_observados) AS viajes_emitidos,
        FROM gold.od_horaria GROUP BY nivel, fecha, origen
    ), recibidos AS (
        SELECT nivel, fecha, destino AS zona, sum(viajes_observados) AS viajes_recibidos,
        FROM gold.od_horaria GROUP BY nivel, fecha, destino
    )
    SELECT coalesce(e.nivel,r.nivel) nivel, coalesce(e.fecha,r.fecha) fecha,
           coalesce(e.zona,r.zona) zona, e.viajes_emitidos, r.viajes_recibidos,
    FROM emitidos e FULL JOIN recibidos r USING(nivel,fecha,zona)""")
display(con.sql("""SELECT g.*, z.nombre FROM gold.movilidad_diaria_zona g
    LEFT JOIN gold.zonas z USING(nivel,zona)
    WHERE g.nivel='municipios' ORDER BY viajes_emitidos DESC LIMIT 10""").df())


,nivel,fecha,zona,viajes_emitidos,viajes_recibidos,nombre
0,municipios,2023-01-01,28079,6.180440e+06,6.324676e+06,Madrid
1,municipios,2023-01-01,08019,3.176260e+06,3.217096e+06,Barcelona
2,municipios,2023-01-01,46250,1.476655e+06,1.518397e+06,València
3,municipios,2023-01-01,41091,1.348057e+06,1.366208e+06,Sevilla
4,municipios,2023-01-01,29067,1.261657e+06,1.270397e+06,Málaga
5,municipios,2023-01-01,50297,1.257681e+06,1.265319e+06,Zaragoza
6,municipios,2023-01-01,30030,1.059194e+06,1.059657e+06,Murcia
7,municipios,2023-01-01,07040,9.344707e+05,9.444047e+05,Palma
8,municipios,2023-01-01,35016,9.280812e+05,9.304632e+05,"Palmas de Gran Canaria, Las"
9,municipios,2023-01-01,48020,7.362710e+05,7.385023e+05,Bilbao


## Ejemplo: perfil horario

In [43]:
display(con.sql("""WITH diario AS (
    SELECT nivel, fecha, hora, sum(viajes_observados) AS viajes
    FROM gold.od_horaria GROUP BY nivel,fecha,hora
)
SELECT nivel, isodow(fecha) AS dia_semana_iso, hora,
       count(*) AS dias_con_filas, count(viajes) AS dias_con_medida,
       avg(viajes) AS media_viajes_observados
FROM diario WHERE nivel='municipios' GROUP BY ALL ORDER BY dia_semana_iso,hora""").df())

,nivel,dia_semana_iso,hora,dias_con_filas,dias_con_medida,media_viajes_observados
0,municipios,7,0,1,1,2.867916e+06
1,municipios,7,1,1,1,3.477599e+06
2,municipios,7,2,1,1,3.528120e+06
3,municipios,7,3,1,1,2.710128e+06
4,municipios,7,4,1,1,2.065118e+06
5,municipios,7,5,1,1,1.725631e+06
6,municipios,7,6,1,1,2.207999e+06
7,municipios,7,7,1,1,2.436104e+06
8,municipios,7,8,1,1,2.258991e+06
9,municipios,7,9,1,1,2.657857e+06


## Datos para un mapa

In [44]:
con.execute("""CREATE OR REPLACE VIEW gold.mapa_movilidad_diaria AS
    SELECT m.*, z.nombre, z.en_catalogo, z.geometria, z.geometria_valida, z.centroide
    FROM gold.movilidad_diaria_zona m JOIN gold.zonas z USING(nivel, zona)""")
display(con.sql("""SELECT nivel, count(*) filas_zona_dia,
    count(*) FILTER (WHERE geometria IS NOT NULL) con_geometria
    FROM gold.mapa_movilidad_diaria GROUP BY nivel""").df())
con.close()

,nivel,filas_zona_dia,con_geometria
0,municipios,2673,2672
1,GAU,2141,2140
2,distritos,3846,3845
